<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/12-case-studies-and-capstone/02-contract-extraction-pipeline-vs-agent.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 案例研究 B —— 合同提取：pipeline 与 agent 之争

**目标：** 在一个真实任务（从合同里抽出字段）上，处理面试官最爱问的那个判断（你*可以*把它做成 agent，但应该吗？），并且用**评测 + 成本数字**来定论，而不是观点。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。

> **先把论点摆出来：** 当步骤事先已知时，**pipeline** 胜过 **agent**：它更便宜、可测试、确定性。agent 那种「自己决定下一步」的自由是你付钱买来的负担，不是资产。这篇 notebook 在同一个任务上把*两者*都建出来，并度量差异。（第 [05/03](../05-agents/03-guardrails-and-budgets.zh.ipynb) 节给了规则；这里我们来证明它。）

## 准备

自包含。在 [console.groq.com](https://console.groq.com/) 领免费 key；在 Colab 里通过**钥匙图标** → secret `GROQ_API_KEY` 添加。（完整走查：[00-setup](../00-setup/00-environment.zh.ipynb)。）

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main"

In [ ]:
from aien import setup
client, MODEL = setup()

## 任务

一个法务运营团队递给你一叠供应商合同，想从每份里抽出四个字段：**parties**（双方）、**effective date**（生效日期）、**annual value**（年度金额）和 **termination notice (days)**（终止通知天数）。同样四个字段，每份合同都要，几千份。

三份干净的合同会让两种方案都显得完美，所以下面这份语料是 20 份合同，分五组，每组都标上它包含的那类混乱。这些标签后面很重要：它们让我们能问每种方案在*哪里*赢，而不只是平均哪个赢。

| 标签 | 难在哪 |
|---|---|
| `clean` | 没什么难的；基线 |
| `date` | 日期有五种不同格式 |
| `words` | 金额用文字写出 |
| `math` | 年度金额没有写出来，必须计算（按月、按季度、基础费 + 支持费） |
| `buried` | 正确的数字挨着一堆干扰项（滞纳金、保险上限、已被取代的价格、付款条款） |

`math` 和 `buried` 两组故意选在 agent 的主场：它拿到了 pipeline 没有的计算器和搜索工具。如果 agent 要在哪一处值回成本，就是这里。

In [ ]:
CONTRACTS = [
    # 干净
    {"id": "C1", "tag": "clean", "text": "This Agreement is made between Acme Corp and Globex LLC, effective January 5, 2024. "
        "The total annual fee is $120,000. Either party may terminate with 30 days written notice."},
    {"id": "C2", "tag": "clean", "text": "Effective 2023-11-15, Initech and Umbrella Inc agree to the terms herein. "
        "Annual contract value: $85,500. Termination requires 60 days notice."},
    {"id": "C3", "tag": "clean", "text": "Between Wayne Enterprises and Stark Industries. Commencement date: March 1 2024. "
        "Yearly amount of $250,000. This contract may be ended upon 90 days notice."},
    {"id": "C4", "tag": "clean", "text": "This Services Agreement is entered into by Hooli Inc and Pied Piper LLC as of June 30, 2024. "
        "Fees: $48,000 per year. Termination: 45 days written notice."},
    {"id": "C5", "tag": "clean", "text": "Soylent Corp and Tyrell Corporation enter this agreement effective 2024-02-29. "
        "Annual fee $310,000. Either party may terminate on 120 days notice."},
    # 日期格式
    {"id": "D1", "tag": "date", "text": "Agreement between Cyberdyne Systems and Weyland-Yutani Corp. Effective date: 07/04/2024 (MM/DD/YYYY). "
        "Annual value $64,000. Termination with 30 days notice."},
    {"id": "D2", "tag": "date", "text": "Massive Dynamic and Aperture Science sign this agreement, effective the 1st of August, 2023. "
        "The yearly fee is $99,000. Notice of termination: 15 days."},
    {"id": "D3", "tag": "date", "text": "Effective 12 Sept 2024, Gringotts Bank and Ollivanders Ltd agree as follows. "
        "Annual amount: $27,500. 60 days notice to terminate."},
    {"id": "D4", "tag": "date", "text": "Dated as of 2024.10.01, between Oscorp Industries and Daily Bugle LLC. "
        "Annual fee of $150,000. Termination upon 30 days written notice."},
    # 金额用文字表示
    {"id": "W1", "tag": "words", "text": "Between Vandelay Industries and Kramerica LLC, effective March 15, 2024. "
        "Annual fee: two hundred fifty thousand dollars. Either party may terminate with sixty (60) days notice."},
    {"id": "W2", "tag": "words", "text": "Prestige Worldwide and Dunder Mifflin agree, effective January 1, 2024. "
        "The annual contract value is one hundred twenty thousand dollars. Termination requires thirty days written notice."},
    {"id": "W3", "tag": "words", "text": "Effective May 20, 2024, Bluth Company and Sitwell Enterprises enter this agreement. "
        "Yearly fee of eighty-five thousand five hundred dollars. Termination on ninety days notice."},
    # 年度金额必须计算
    {"id": "M1", "tag": "math", "text": "Agreement between Nakatomi Trading and Genco Olive Oil, effective April 1, 2024. "
        "Fees are $10,000 per month. Termination: 30 days notice."},
    {"id": "M2", "tag": "math", "text": "Effective 2024-01-15, Monarch Solutions and Pendant Publishing agree. "
        "The vendor will be paid $5,250 per month. Either party may terminate on 45 days notice."},
    {"id": "M3", "tag": "math", "text": "Between Krusty Krab LLC and Chum Bucket Inc, effective November 1, 2023. "
        "Quarterly fees of $30,000. Termination requires four weeks notice."},
    {"id": "M4", "tag": "math", "text": "Contoso Ltd and Fabrikam Inc, effective June 1, 2024. "
        "Fees: a base fee of $40,000 per year plus support at $2,000 per month. Notice period: 60 days."},
    # 正确数字挨着干扰项
    {"id": "B1", "tag": "buried", "text": "MASTER SERVICES AGREEMENT. 1. Definitions. Affiliate means any entity controlling or controlled by a party. "
        "2. Term. This Agreement is effective as of February 10, 2024 between Globo Gym Corp (Client) and Average Joes LLC (Vendor). "
        "3. Fees. Client shall pay Vendor an annual fee of $72,000. Late payments accrue interest at 1.5% per month, and a $500 late fee "
        "applies after 10 days. 4. Insurance. Vendor shall maintain coverage of at least $1,000,000. 5. Termination. Either party may "
        "terminate this Agreement upon 90 days written notice. Sections 4 and 6 survive termination for 3 years."},
    {"id": "B2", "tag": "buried", "text": "This Agreement between Spacely Sprockets and Cogswell Cogs is effective January 10, 2024. "
        "The proposal dated November 2, 2023 is superseded. The previous annual fee of $40,000 under the prior agreement no longer applies; "
        "the annual fee under this Agreement is $46,000. Termination: 30 days notice. Cure period: 15 days."},
    {"id": "B3", "tag": "buried", "text": "Between Planet Express Inc and Moms Friendly Robot Co. Effective date: August 8, 2024. "
        "Payment terms: net 30. Annual fee: $33,000, payable in 12 monthly installments of $2,750. Termination: 14 days written notice."},
    {"id": "B4", "tag": "buried", "text": "This agreement is between Los Pollos Hermanos LLC (hereinafter Customer) and Madrigal Electromotive GmbH "
        "(hereinafter Supplier), effective as of 1 May 2024. Supplier quoted $25,000 in its 2023 bid; the agreed price is $18,000 annually. "
        "Invoices are due within 45 days. Either party may terminate upon sixty days notice."},
]

def g(parties, date, value, notice):
    return {"parties": parties, "effective_date": date, "annual_value": value, "termination_notice_days": notice}

GOLD = {
    "C1": g(["Acme Corp", "Globex LLC"], "2024-01-05", 120000, 30),
    "C2": g(["Initech", "Umbrella Inc"], "2023-11-15", 85500, 60),
    "C3": g(["Wayne Enterprises", "Stark Industries"], "2024-03-01", 250000, 90),
    "C4": g(["Hooli Inc", "Pied Piper LLC"], "2024-06-30", 48000, 45),
    "C5": g(["Soylent Corp", "Tyrell Corporation"], "2024-02-29", 310000, 120),
    "D1": g(["Cyberdyne Systems", "Weyland-Yutani Corp"], "2024-07-04", 64000, 30),
    "D2": g(["Massive Dynamic", "Aperture Science"], "2023-08-01", 99000, 15),
    "D3": g(["Gringotts Bank", "Ollivanders Ltd"], "2024-09-12", 27500, 60),
    "D4": g(["Oscorp Industries", "Daily Bugle LLC"], "2024-10-01", 150000, 30),
    "W1": g(["Vandelay Industries", "Kramerica LLC"], "2024-03-15", 250000, 60),
    "W2": g(["Prestige Worldwide", "Dunder Mifflin"], "2024-01-01", 120000, 30),
    "W3": g(["Bluth Company", "Sitwell Enterprises"], "2024-05-20", 85500, 90),
    "M1": g(["Nakatomi Trading", "Genco Olive Oil"], "2024-04-01", 120000, 30),
    "M2": g(["Monarch Solutions", "Pendant Publishing"], "2024-01-15", 63000, 45),
    "M3": g(["Krusty Krab LLC", "Chum Bucket Inc"], "2023-11-01", 120000, 28),
    "M4": g(["Contoso Ltd", "Fabrikam Inc"], "2024-06-01", 64000, 60),
    "B1": g(["Globo Gym Corp", "Average Joes LLC"], "2024-02-10", 72000, 90),
    "B2": g(["Spacely Sprockets", "Cogswell Cogs"], "2024-01-10", 46000, 30),
    "B3": g(["Planet Express Inc", "Moms Friendly Robot Co"], "2024-08-08", 33000, 14),
    "B4": g(["Los Pollos Hermanos LLC", "Madrigal Electromotive GmbH"], "2024-05-01", 18000, 60),
}
assert set(GOLD) == {c["id"] for c in CONTRACTS}
from collections import Counter
print(f"{len(CONTRACTS)} contracts, 4 fields each:", dict(Counter(c["tag"] for c in CONTRACTS)))


## 共用一个打分器、一个成本计、一套 schema

两种方案都先要有评测和计量，这样「哪个更好」就是一个数字，而不是感觉。我们对四个字段做精确匹配打分，并统计 token（Groq 免费档是 $0，但 token 是诚实的成本代理：与 [01/04](../01-model-apis/04-context-and-caching.zh.ipynb) 和 [05/03](../05-agents/03-guardrails-and-budgets.zh.ipynb) 同一个思路）。

两种方案拿到**同一份字段规格**（`FIELDS`）、同一个模型、同样的 `max_tokens` 和 `temperature=0`。唯一不同的是谁掌握控制流：代码，还是模型。

In [ ]:
import json, re

FIELDS = ("{parties:[the two party names], effective_date:'YYYY-MM-DD', "
          "annual_value:<total value per year in whole dollars, integer>, termination_notice_days:<integer>}")
MAX_TOKENS = 1024   # 两者预算相同；gpt-oss 会把其中一部分花在推理上

def parse_json(text):
    m = re.search(r'\{.*\}', text or "", re.DOTALL)
    try: return json.loads(m.group()) if m else {}
    except Exception: return {}

def as_int(v):
    try: return int(str(v).replace(",", "").replace("$", "").strip())
    except (TypeError, ValueError): return None

def score(pred, gold):
    """四个字段中精确匹配的比例（轻量归一化：大小写、'$'、','）。"""
    ok = 0
    for k in gold:
        v = pred.get(k)
        if k == "parties":
            ok += isinstance(v, list) and sorted(str(p).strip().lower() for p in v) == sorted(p.lower() for p in gold[k])
        elif k == "effective_date":
            ok += str(v).strip() == gold[k]
        else:
            ok += as_int(v) == gold[k]
    return ok / len(gold)

class Meter:
    """累计跨调用的 token 用量，作为 $ 和延迟的替身。"""
    def __init__(self): self.calls = 0; self.tokens = 0
    def add(self, resp): self.calls += 1; self.tokens += resp.usage.total_tokens
    def __str__(self): return f"{self.calls} calls, {self.tokens} tokens"


## 方案 1 —— agent

诱人的构建方式：给模型工具，让它*自己决定*怎么抽每个字段。感觉灵活又「聪明」。我们把 [05/01](../05-agents/01-agent-loop-from-scratch.zh.ipynb) 的原生循环接上两个工具，`find_line`（搜索合同）和 `calculate`（精确算术），让它来驱动。计算器在 `math` 组是实打实的优势：pipeline 得在心里做那套算术。

In [ ]:
import ast

# agent 可以调用的工具。故意做得通用：由 agent 决定怎么用它们。
def find_line(contract_text, keyword):
    hits = [ln for ln in re.split(r'(?<=[.])\s+', contract_text) if keyword.lower() in ln.lower()]
    return " | ".join(hits[:3]) if hits else "(no line matches)"

def calculate(expression):
    """对数字做精确算术，支持 + - * / 和括号。其他任何东西都不会被求值。"""
    try:
        tree = ast.parse(expression, mode="eval")
    except SyntaxError:
        return "error: not an arithmetic expression"
    ok = (ast.Expression, ast.BinOp, ast.UnaryOp, ast.Add, ast.Sub, ast.Mult, ast.Div, ast.USub)
    for n in ast.walk(tree):
        if not (isinstance(n, ok) or (isinstance(n, ast.Constant) and isinstance(n.value, (int, float)))):
            return "error: only numbers and + - * / are allowed"
    try:
        result = eval(compile(tree, "<calc>", "eval"))
    except ZeroDivisionError:
        return "error: division by zero"
    return str(int(result)) if float(result).is_integer() else str(result)

TOOLS = [
    {"type": "function", "function": {"name": "find_line",
        "description": "Return up to 3 sentences in the contract containing a keyword (e.g. 'terminate', 'effective', 'fee').",
        "parameters": {"type": "object", "properties": {"keyword": {"type": "string"}}, "required": ["keyword"]}}},
    {"type": "function", "function": {"name": "calculate",
        "description": "Evaluate an arithmetic expression exactly, e.g. '5250 * 12' or '40000 + 2000 * 12'.",
        "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}}},
]

def extract_agent(contract, meter, max_turns=8):
    text = contract["text"]
    impls = {"find_line": lambda a: find_line(text, a.get("keyword", "")),
             "calculate": lambda a: calculate(a.get("expression", ""))}
    messages = [
        {"role": "system", "content": "Extract contract fields. Use find_line to locate info and calculate for any "
                                       "arithmetic. When done, reply with ONLY a JSON object: " + FIELDS},
        {"role": "user", "content": f"Contract:\n{text}"},
    ]
    for _ in range(max_turns):
        resp = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS,
                                              max_tokens=MAX_TOKENS, temperature=0)
        meter.add(resp)
        msg = resp.choices[0].message
        if resp.choices[0].finish_reason != "tool_calls":
            return parse_json(msg.content)
        messages.append(msg)
        for tc in msg.tool_calls:
            try:
                args = json.loads(tc.function.arguments or "{}")
                fn = impls.get(tc.function.name)
                out = fn(args) if fn else f"error: unknown tool {tc.function.name}"
            except Exception as e:
                out = f"error: {e}"
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})
    return {}   # 轮次用尽：算作未命中，就像真实的预算截断

print("agent ready:", [t["function"]["name"] for t in TOOLS])


> **⚠️ 生产现实 ——** agent 的每一次运行可能花掉*不同*的轮数和工具调用数。这就是内建的非确定性：两份一模一样的合同可能花费不同，甚至抽取出不同结果。注意看 token 数；它比你猜的更高，因为每一轮都会重发整段历史（[01/04](../01-model-apis/04-context-and-caching.zh.ipynb)）。

## 方案 2 —— pipeline

步骤是已知的：对*每一份*合同我们都要同样四个字段。那就把步骤写下来。一次模型调用、固定 schema、确定性解析：这就是 [01/01 结构化输出](../01-model-apis/01-structured-output.zh.ipynb) 的模式。没有循环、没有工具选择、没有重发历史，也没有计算器：模型必须自己把 `math` 组做对。

In [ ]:
def extract_pipeline(contract, meter):
    # 一次调用，一件事：返回固定 schema。所谓「步骤」就是 schema 本身。
    resp = client.chat.completions.create(
        model=MODEL, max_tokens=MAX_TOKENS, temperature=0,
        messages=[{"role": "system", "content":
                   "Extract these fields from the contract and reply with ONLY a JSON object: " + FIELDS},
                  {"role": "user", "content": contract["text"]}])
    meter.add(resp)
    return parse_json(resp.choices[0].message.content)

print("pipeline ready")


## 两种方案各跑几次

一个非确定性系统只跑一次就是个轶事。我们让每份合同都过两种方案各 `REPEATS` 次，并保留每一个结果，这样下面的结论能报出一个区间，而不是一个侥幸的数字。

> **🔵 撞上速率限制？** 这里是约 20 × 2 × `REPEATS` 次抽取，而且 agent 每份合同要发好几次调用。如果拿到 `429`，直接**重跑这个单元格**：它会跳过已完成的工作并接着跑。或者调小 `REPEATS`，或像 [02/01](../02-evals-basics/01-measuring-outputs.zh.ipynb) 那样换模型：`client, MODEL = setup(model='openai/gpt-oss-20b')`。

In [ ]:
REPEATS = 3

results = globals().get("results", [])          # 可安全重跑：接着跑，而不是从头开始
done = {(r["approach"], r["run"], r["id"]) for r in results}
APPROACHES = {"pipeline": extract_pipeline, "agent": extract_agent}

for run in range(REPEATS):
    for c in CONTRACTS:
        for name, fn in APPROACHES.items():
            if (name, run, c["id"]) in done:
                continue
            m = Meter()
            pred = fn(c, m)
            results.append({"approach": name, "run": run, "id": c["id"], "tag": c["tag"],
                            "score": score(pred, GOLD[c["id"]]), "calls": m.calls, "tokens": m.tokens, "pred": pred})
    print(f"run {run + 1}/{REPEATS} done ({len(results)} results)")


## 结论 —— 同一个任务，度量出来的

In [ ]:
from statistics import mean

def rows(approach, **where):
    return [r for r in results if r["approach"] == approach and all(r[k] == v for k, v in where.items())]

summary = {}
print(f"{'':10} {'accuracy':>9} {'range over runs':>16} {'calls/contract':>15} {'tokens/contract':>16}")
for a in APPROACHES:
    per_run = [mean(r["score"] for r in rows(a, run=k)) for k in range(REPEATS) if rows(a, run=k)]
    rs = rows(a)
    summary[a] = {"acc": mean(per_run), "tokens": mean(r["tokens"] for r in rs)}
    print(f"{a:10} {mean(per_run):>9.0%} {min(per_run):>8.0%} .. {max(per_run):<5.0%} "
          f"{mean(r['calls'] for r in rs):>15.1f} {summary[a]['tokens']:>16.0f}")

ratio = summary["agent"]["tokens"] / max(summary["pipeline"]["tokens"], 1)
print(f"\nthe agent spent {ratio:.1f}x the tokens of the pipeline; "
      f"accuracy {summary['agent']['acc']:.0%} (agent) vs {summary['pipeline']['acc']:.0%} (pipeline).")


## agent 应该在哪里赢？按组看准确率

平均值把有意思的部分藏起来了。按标签拆开：在 `clean` 和 `date` 上，两种方案都应该接近上限，所以那里任何 token 差异都是纯粹的开销。真正的考验是 `math` 和 `buried`，也就是 agent 拥有 pipeline 所缺工具的地方。

把这张表当成一张价目表来读：**对每一组，agent 多花了多少 token，又买回了多少准确率？** 某一组里 agent 明显赢，就是把*那一步*提升为工具调用的证据（见下面「agent 什么时候*确实*对」）。它不是把整件事做成 agent 的证据。

In [ ]:
tags = list(dict.fromkeys(c["tag"] for c in CONTRACTS))
print(f"{'group':8} {'n':>3} {'pipeline acc':>13} {'agent acc':>10} {'pipeline tok':>13} {'agent tok':>10}")
for t in tags:
    p, a = rows("pipeline", tag=t), rows("agent", tag=t)
    n = len({r["id"] for r in p})
    print(f"{t:8} {n:>3} {mean(r['score'] for r in p):>13.0%} {mean(r['score'] for r in a):>10.0%} "
          f"{mean(r['tokens'] for r in p):>13.0f} {mean(r['tokens'] for r in a):>10.0f}")

# 那些错误本身就是最有用的输出：每种方案各自错在哪？
# 用 ascii() 打印期望值与返回值，这样长得像的字符（Unicode 连字符、
# 弯引号）会以 \u 转义的形式暴露出来：精确匹配的打分器也会把它们算作未命中。
print("\nfields missed (any run):")
for c in CONTRACTS:
    for a in APPROACHES:
        bad = {r["run"]: r["pred"] for r in rows(a, id=c["id"]) if r["score"] < 1}
        if bad:
            run, pred = next(iter(bad.items()))
            wrong = [k for k in GOLD[c["id"]] if score({k: pred.get(k)}, {k: GOLD[c["id"]][k]}) < 1]
            print(f"  {c['id']} [{c['tag']}] {a:8} missed {wrong} in {len(bad)}/{REPEATS} runs")
            for k in wrong:
                print(f"      expected {ascii(GOLD[c['id']][k])}")
                print(f"      returned {ascii(pred.get(k))}")


## 同样的输入，同样的输出？

对于测试和值班来说，问题不只是「多准」，还有「我再跑一次，会得到同样的东西吗？」在 `temperature=0` 下，你会希望两者都稳定。数一数有多少份合同在多次运行中某个方案返回了*不同*的答案，以及同一份合同的成本波动了多大。

In [ ]:
for a in APPROACHES:
    changed, spreads = [], []
    for c in CONTRACTS:
        rs = rows(a, id=c["id"])
        if len({json.dumps(r["pred"], sort_keys=True) for r in rs}) > 1:
            changed.append(c["id"])
        toks = [r["tokens"] for r in rs]
        spreads.append(max(toks) - min(toks))
    print(f"{a:8}: {len(changed)}/{len(CONTRACTS)} contracts changed answer across runs {changed}; "
          f"max token swing on one contract: {max(spreads)}")


> **🚩 常见错误 ——** 已经上线的「agent」里有很大一部分是披着风衣的 pipeline：模型在一些永远按同样顺序执行的步骤之间「做决定」，而团队为 pipeline 的行为付着 agent 的代价（非确定性、按步计费、评测困难、重发历史）。提取就是教科书级的例子：四个字段是固定的，所以那些「决定」是假的。

> **🔵 面试信号 ——**「我们把它做成 pipeline，因为步骤是已知的，这是它与 agent 版本的 token/准确率对比」比「我们做了个 agent」是*更强*的回答。它表明你知道**什么时候不该**伸手去用循环，这正是第 [05/03](../05-agents/03-guardrails-and-budgets.zh.ipynb) 节讲的判断。

## agent 什么时候*确实*对

这个案例主张 pipeline，**是因为步骤是已知的**。把这一点反过来，agent 就赢了：

- **路径取决于你发现什么。**「读这个错误，决定接下来检查什么，重复」是调试或研究类任务，第 N+1 步在拿到第 N 步的返回之前真的不知道。
- **开放式工具使用。** 动作集合及其顺序会随输入变化，无法穷举。
- **pipeline 里的一个分支。** 通常正确的形态是*一条 pipeline* 加上一个小的*有界 agent*，只提升给那个真正分叉的步骤，而不是用 agent 包住整件事。

> **⭐ 关键收获 ——**「pipeline vs agent」不在于是否高级；而在于步骤能否事先知道。已知 → pipeline（更便宜、可测试、确定性）。真正动态 → 一个*仅限那部分*的有界 agent。从 pipeline 起步；当你有了证据表明某一步需要 agent 时，再把它提升为 agent。反过来迁移——把一个不稳定的生产系统去 agent 化——痛苦得多。

## 练习

1. **让 agent 显得好看。** 构造一份合同，让 pipeline 的单次调用*失败*而 agent 能救回来（比如一份很长的合同，费用出现在附录里）。把它加进 `CONTRACTS`，给一个自己的标签，然后重跑。agent 真赢了吗，代价是多少 token？这才是对「循环什么时候值回价格」的诚实检验。
2. **提升一个步骤。** 保留 pipeline，但用*单个*有界步骤处理 `math` 组：让模型给出原始金额和它的周期（`{"amount": 5250, "per": "month"}`），乘法在代码里做。这个混合方案能否以 pipeline 的成本达到 agent 的准确率？
3. **规模化的成本。** 用上面的 token 数，估算每月 10 万份合同时的成本差异（从 [01/04](../01-model-apis/04-context-and-caching.zh.ipynb) 取一个真实的 $/token）。写出你会放进设计文档、用来支撑这个选择的那一句话。
4. **更小的模型。** 用 `setup(model='openai/gpt-oss-20b')` 重跑（先清空 `results`）。模型更弱时，两种方案的差距是变大还是变小，落在哪一组？